# **Creating a RAG with Local Ollama and LangChain [Countries Wiki Data]**





*This is a notebook for the Coursera course, Building and Deploying Generative AI Models from the specialization Generative AI Fundamentals by Alberta Machine Intelligence Institute (Amii) and is under an MIT License.*

Author: Anahita Doosti

Github Repository: [link](https://github.com/anna1995d/Building-and-Deploying-Generative-AI-Models-RAG-Notebooks)


---



In this notebook, we will use LangChain to upgrade our pipeline from before. This will make it easier to add new components.


In [ ]:
# setting up command-line operation support
# xterm allows us to run shell commands directly in a Jupyter notebook
# Uncomment the appropriate section based on your setup

# GOOGLE COLAB
!pip install colab-xterm
%load_ext colabxterm

# JUPYTER LAB
# !pip install jupyterlab-ptyproc
# %load_ext notebook_xterm

We need Xterm so we can run the Ollama server in the background.

*The following sets up our GPU. Please ignore, if you are not using Vertex AI's Colab Enterprise.*

In [ ]:
# @title
# # @title GPU setup
# !sudo apt-get install lshw
# !curl https://ollama.ai/install.sh | sh
# !pip install ollama

# !echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
# !sudo apt-get update && sudo apt-get install -y cuda-drivers

# import os
# # Set LD_LIBRARY_PATH so the system NVIDIA library
# os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

In [ ]:
%xterm
# Type in the following in the terminal:
# (INSTALL) curl https://ollama.ai/install.sh | sh
# (RUN) ollama serve &

*If at any later point you get the error message "Failed to connect to Ollama", rerun the cell calling xterm and run the serve command again.*

In [ ]:
!ollama pull gemma3:1b
!ollama pull embeddinggemma:300m


In [ ]:
!ollama list

In [ ]:
# install the LangChain python libraries
!pip install --quiet --upgrade langchain
!pip install --quiet --upgrade langchain_core
!pip install --quiet --upgrade langchain-ollama
!pip install --quiet --upgrade langchain-text-splitters langchain-community

# Load the RAG data

Our dataset consists of 10 documents, each containing cleaned text from a Wikipedia article on a different country. It also includes a (.csv) file containing question and answer pairs based only on these documents.

This dataset is a processed subset of a larger one that you can find linked below:


*Smith, N. A., Heilman, M., & Hwa, R. (2008, September) [[link]](https://https://www.kaggle.com/datasets/rtatman/questionanswer-dataset/data)*

In [ ]:
# Uncomment if you need to delete from another run
# !rm -rf small_country_qa_dataset

Using the the code below, you can download the dataset from the url and unzip it. The list `articles_filenames` contrains the path for each (.txt) article file and `qa_eval_filename` contains the path to the (.csv) Q&A file.

In [ ]:
# Download, and extract the database
import io       # For creating an in-memory binary stream
import zipfile  # For handling zip files
import requests # For loading the file from the web

# The database is a .zip file containing .txt articles
url = "https://drive.google.com/uc?export=view&id=1Icqw0yYzmsT0CyrcTJ3UNMY9icnOQcNI"

try:
    db_zip = requests.get(url, timeout=60)
    db_zip.raise_for_status() # fail fast on bad status

    filenames = list()

    with zipfile.ZipFile(io.BytesIO(db_zip.content)) as zf:
        filenames = zf.namelist()
        zf.extractall() # extract all files in the local directory

    print(f"Downloaded {len(filenames)} files.")

except requests.exceptions.RequestException as e:
    print(f"Error downloading the file: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# Separate eval data and articles
articles_filenames = filenames[-10:] # Feel free to limit to fewer files if it takes too long
qa_eval_filename =  filenames[0]



Given the file name, the next function will return the file header (containing the article subject) and the list of all sentences in the file.
This easy approach will be our initial chunking strategy.

In [ ]:
def extract_sentences(filename):
    with open(filename, 'r', encoding="utf-8") as file:
        subject = file.readline().rstrip('\n\r') # separate the file header
        file_content = file.read().rstrip('\n\r').replace('\n', '').split('.') # read the rest of the file

        return subject, file_content

# Models and Vector Store Initialization

With Ollama, we used to pass the model (downloaded via the `pull` command) name using methods like `ollama.embed()` and `ollama.generate()`. If we ever want to use a different model outside of Ollama (e.g. OpenAI models via API), we would need to rewrite everything to work with the new model's API.

One of LangChain's core components is Models. Models are object classes that allow us to encapsulate our model, so that the pipeline operates the same way regardless of where the model's from. This way we can easily switch between different models by only changing the initialization.

On top of this, LangChain offers a variety of integrations from different model providers such as Ollama, OpenAI, HuggingFace and more. You can find all supported providers [here](https://docs.langchain.com/oss/python/integrations/providers/all_providers).

For Ollama, we have `ChatOllama()` and `OllamaEmbeddings()` which we can import form `langchain_ollama`.

In [ ]:
# Set up the Ollama models
from langchain_ollama import ChatOllama

llm_model = ChatOllama(model="gemma3:1b",
                 temprature=0)

# Set up the Ollama embedding model
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(model="embeddinggemma:300m")

Next, let's define the vector store. This is another core component of LangChain. For now we use the `InMemoryVectorStore` from `langchain_core.vectorstores`. The vector store object initializer, expects an embedding model as input. This model is used to vectorize any chunk that is added to the vector store.

As with the model class, we can easily change our vector store by using a different initializer. We'll explore vectors further in the next video.

In [ ]:
# Set up vector storing in memory
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding_model)

# Populate the Vector Store

Now, we chunk the data and add it to `vector_store`. The vector store automatically embeds the chunks using the embedding model.

## LangChain `Document`

An important object in LangChain is a `Document`. Previously, we use this only for the source documents in the dataset.
However, in LangChain, `Document` is an abstraction that represents a unit of text and its associated metadata. Often, this refers to a chunk of text.

A `Document` has 3 attributes:


*   `page_content: str`
*   `metadata: dict`
*   `id (optional): str`

We'll use `metadata` to track the information we previously stored in `Chunk`.



In [ ]:
from importlib.metadata import metadata
from langchain_core.documents import Document

# Extract sentences from every file and populate the database
# If it's too slow for you to process every file, you can limit the number of
# files by changing the range in the for loop or modifying filenames
all_docs = list()
for filename in articles_filenames:
    subject, file_content = extract_sentences(filename)

    for chunk in file_content:
        if chunk.strip() == '':
            continue
        chunk += '.'
        all_docs.append(Document(page_content=chunk,
                                 metadata={"source": filename,
                                           "subject":subject}))
    print(f"Split {filename} into chunks.")

_ = vector_store.add_documents(all_docs)
print(f"Added {len(all_docs)} documents to the vector store.")

# Create LangChain `Retriever`

In [ ]:
# Create a retriever from vector store
retriever = vector_store.as_retriever(search_type="similarity",
                                      search_kwargs={"k": 5})

# Define Prompt Templates

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = ("You are a helpful question-answering chatbot. \n"
                  "Use only the following pieces of context to answer the question. \n"
                  "If you don't know the answer, just say that you don't know and don't make up any new information.")

USER_TEMPLATE = ("Here is the context: \n{context_string} \n"
                    "Question: {question}\n\n")
                    # "Answer:")

PROMPT_TEMPLATE = ChatPromptTemplate([
    ("system", SYSTEM_PROMPT),
    ("human", USER_TEMPLATE)
])

# Retrieve and Generate


In [ ]:
def answer_query(input_query, verbose=False):
    # Retrieve the top 5 relevant chunks
    retrieved_docs = retriever.invoke(input_query)
    # Join context chunks in one string
    context_string = '\n'.join([doc.page_content for doc in retrieved_docs])

    # Fill in the prompt templates with the question and context
    chat_prompt = PROMPT_TEMPLATE.format_messages(question=input_query,
                                         context_string=context_string)

    # Generate response
    response = llm_model.invoke(chat_prompt)

    if verbose:
        print("Chatbot response:")
        print(response.content)
        print("------------------------------------------")
        print(f'Retrieved context:')
        for n, doc in enumerate(retrieved_docs):
            print(f"* {n+1} \t Subject: {doc.metadata['subject']} - {doc.page_content}")
    else:
        return response.content, retrieved_docs

Let's try it!

In [ ]:
answer_query("What is the life expectancy for men in Finland?", verbose=True)

# Evaluation

Like with any ML pipeline, we need to examine the results to evaluate quality. Here, we will try a few questions that we know the answer to. You can experiment with different questions and top K values.

## Question 1
**What is Canada's one significant non-official language?**

In the Canada article, we can find:


> Some significant non-official first languages include Chinese (853,745 first-language speakers), Italian (469,485), German (438,080), and Punjabi (271,220).

RAG results:

In [ ]:
answer_query("What is Canada's one significant non-official language?", verbose=True)

## Question 2
**Who is Canada's richest man?**

This last question cannot be answered by any information in the database. Based on our instruction prompt, our expectation is for the model to indicate exactly that.

In [ ]:
answer_query("Who is Canada's richest man?", verbose=True)

## Question 3
**What universities are in Education City in Qatar?**

If we look at the article on Qatar we can find the following sections:


> Qatar also established Education City, which consists of international colleges.


> ... some major American universities have opened branch campuses in **Education City**, Qatar. These include Carnegie Mellon University, Georgetown University School of Foreign Service, Texas A&M University, Virginia Commonwealth University, and Cornell University's Weill Medical College.

Now, let's see the RAG results!

In [ ]:
answer_query("What universities are in Education City in Qatar?", verbose=True)

## Evaluation Q&A Data

A few questions are not enough. We need a Q&A dataset to fully evaluate RAG performance.

Below, you will load the Q&A (.csv) file.

In [ ]:
import pandas as pd

# Load evaluation Q&A data as a Pandas dataframe
df = pd.read_csv(qa_eval_filename, sep="\t")
print(f"Number of questions: {len(df)}")
df.head()

Feel free to sample a smaller subset of the eval data using the code below.

In [ ]:
# # Uncomment to run the evaluation with a smaller QA set
# df = df.sample(n=30, random_state=42)
# df.head()

You can run the next cell to compare the RAG answers to the evaluation question with the ground truth. Feel free to adjust the range.

In [ ]:
for i in range(10): #len(df)):
  row = df.iloc[i]
  question = f"For the country of {row['ArticleTitle']}, {row['Question']}"
  print(f"Question {i+1}: {question}")
  print(f"Grounded Answer {i+1}: {row['Answer']}")
  print(f"RAG Answer {i+1}: {answer_query(question)[0]}")
  print("-----------------------------------------------------------------------")

In [ ]:
# Uncomment and run if you want open up space by deleting ollama
# !rm -rf /usr/local/bin/ollama

*MIT License Copyright (c) 2025 Alberta Machine Intelligence Institute (amii)*